# 05 - Model Training (Simple Pipeline)

Loads the fitted selector and final feature list produced by notebook 04, applies transform_selected to test data, tunes XGBoost with Optuna, trains the final model, and saves the model artifact.

In [7]:
# ! python -m pip install optuna

In [8]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import joblib
import json
import config
from src.io import logger
from src.models import build_model, xgb_safe_frame
from src.optimization import optimize_model
from src.feature_selection import transform_selected
from sklearn.metrics import roc_auc_score

## Step 1: Load Preprocessed Training Data

In [9]:
X_train = pd.read_csv(config.PROCESSED_DIR / "X_train_preprocessed.csv", index_col=0)

y_train_df = pd.read_csv(config.PROCESSED_DIR / "y_train.csv")
y_train = y_train_df.iloc[:, 0] if len(y_train_df.columns) == 1 else y_train_df["BCR"]

logger.info(f"Training data loaded: {X_train.shape}, Positives: {int(y_train.sum())}")

2026-09-01 09:54:39 | INFO     | prostate_bcr | Training data loaded: (343, 19018), Positives: 46


## Step 2: Load Artifacts From Notebook 04

Requires `fitted_selector.joblib` and `selected_features_final.csv`. Raises a clear error if notebook 04 has not been run.

In [10]:
selector_path = config.MODELS_DIR / "fitted_selector.joblib"
features_path = config.TABLES_DIR / "selected_features_final.csv"

try:
    fitted_selector = joblib.load(selector_path)
    selected_df = pd.read_csv(features_path)
    final_features = selected_df["feature"].tolist()
    logger.info(f"Loaded {len(final_features)} final features from the pipeline.")
except FileNotFoundError as e:
    raise FileNotFoundError(
        f"{e}\n\n"
        "Please run '04_feature_selection.ipynb' first to generate "
        "'fitted_selector.joblib' and 'selected_features_final.csv'."
    )

2026-09-01 09:54:39 | INFO     | prostate_bcr | Loaded 50 final features from the 3-layer pipeline.


## Step 3: Apply Feature Selection to Training Data

Use transform_selected to ensure training data matches the exact feature set that will be used for test/external data.

In [11]:
# Transform training data using fitted selector
X_train_selected = transform_selected(X_train, fitted_selector)

# Ensure we only keep features in final_features list
available_in_train = [f for f in final_features if f in X_train_selected.columns]
X_train_final = X_train_selected[available_in_train].copy()

# Fill any missing features with 0.0 (should not happen normally)
missing_feats = set(final_features) - set(available_in_train)
if missing_feats:
    logger.warning(f"Missing {len(missing_feats)} features in training data: {list(missing_feats)[:5]}...")
    for feat in missing_feats:
        X_train_final[feat] = X_train_final = X_train_final[final_features]

logger.info(f"Final training matrix shape: {X_train_final.shape}")

2026-09-01 09:54:40 | INFO     | prostate_bcr | Layer 2 - Created 4 clean engineered features
2026-09-01 09:54:40 | INFO     | prostate_bcr | Final training matrix shape: (343, 50)


## Step 4: Hyperparameter Tuning with Optuna

In [12]:
logger.info("Starting Optuna hyperparameter tuning for XGBoost...")
n_trials = config.N_OPTUNA_TRIALS if hasattr(config, "N_OPTUNA_TRIALS") else 100
_, study, best_params = optimize_model(
    model_name="XGBoost",
    X_train=X_train_final,
    y_train=y_train,
    n_trials=n_trials,
    cv_splits=config.INNER_SPLITS,
    scoring="roc_auc",
)

logger.info(f"Best CV AUC: {study.best_value:.4f}")
logger.info(f"Best params: {json.dumps(best_params, indent=2, default=str)}")

pd.DataFrame([best_params]).to_csv(config.TABLES_DIR / "best_hyperparameters.csv", index=False)

2026-09-01 09:54:40 | INFO     | prostate_bcr | Starting Optuna hyperparameter tuning for XGBoost...
d:\Prostate_BCR\venv\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
[I 2026-09-01 09:54:40,078] A new study created in memory with name: XGBoost_BCR_Optimization
2026-09-01 09:54:40 | INFO     | prostate_bcr | Created Optuna study: XGBoost_BCR_Optimization
2026-09-01 09:54:40 | INFO     | prostate_bcr | Starting Optuna optimization for XGBoost: 100 trials
Best trial: 0. Best value: 0.807267:   1%|          | 1/100 [00:02<03:51,  2.33s/it]

[I 2026-09-01 09:54:42,411] Trial 0 finished with value: 0.8072671156004491 and parameters: {'max_depth': 5, 'min_child_weight': 6.351221010640703, 'gamma': 0.007177141927992002, 'learning_rate': 0.030405325392865647, 'n_estimators': 200, 'subsample': 0.662397808134481, 'colsample_bytree': 0.6232334448672797, 'colsample_bylevel': 0.9464704583099741, 'reg_alpha': 0.002570603566117598, 'reg_lambda': 0.023585940584142682}. Best is trial 0 with value: 0.8072671156004491.


Best trial: 0. Best value: 0.807267:   2%|▏         | 2/100 [00:04<03:14,  1.98s/it]

[I 2026-09-01 09:54:44,148] Trial 1 finished with value: 0.7863215488215488 and parameters: {'max_depth': 3, 'min_child_weight': 7.579479953348009, 'gamma': 0.04566054873446119, 'learning_rate': 0.0033572967053517922, 'n_estimators': 250, 'subsample': 0.6733618039413735, 'colsample_bytree': 0.7216968971838151, 'colsample_bylevel': 0.8099025726528951, 'reg_alpha': 7.71800699380605e-05, 'reg_lambda': 4.17890272377219e-06}. Best is trial 0 with value: 0.8072671156004491.


Best trial: 2. Best value: 0.807828:   3%|▎         | 3/100 [00:06<03:52,  2.40s/it]

...

Best trial: 73. Best value: 0.832421:  99%|█████████▉| 99/100 [01:35<00:01,  1.29s/it]

[I 2026-09-01 09:56:15,977] Trial 98 finished with value: 0.7997895622895624 and parameters: {'max_depth': 9, 'min_child_weight': 0.010846286400028328, 'gamma': 3.769633416189945e-05, 'learning_rate': 0.00872373363708596, 'n_estimators': 1000, 'subsample': 0.845742164269861, 'colsample_bytree': 0.784661064209474, 'colsample_bylevel': 0.9029261643700974, 'reg_alpha': 5.1216738521767445e-06, 'reg_lambda': 0.0005068980926364405}. Best is trial 73 with value: 0.8324214365881032.


Best trial: 73. Best value: 0.832421: 100%|██████████| 100/100 [01:36<00:00,  1.04it/s]
2026-09-01 09:56:16 | INFO     | prostate_bcr | Optimization complete! Best roc_auc = 0.8324


[I 2026-09-01 09:56:16,450] Trial 99 finished with value: 0.8032968574635242 and parameters: {'max_depth': 7, 'min_child_weight': 0.013802393682745877, 'gamma': 0.09320746284998274, 'learning_rate': 0.057944485172701364, 'n_estimators': 650, 'subsample': 0.8551357752317479, 'colsample_bytree': 0.9133576437682833, 'colsample_bylevel': 0.8346893338126792, 'reg_alpha': 5.087727038556954e-06, 'reg_lambda': 4.353747027771329e-06}. Best is trial 73 with value: 0.8324214365881032.


2026-09-01 09:56:16 | INFO     | prostate_bcr | Best CV AUC: 0.8324
2026-09-01 09:56:16 | INFO     | prostate_bcr | Best params: {
  "max_depth": 7,
  "min_child_weight": 0.0462919264399123,
  "gamma": 1.510789381676131e-08,
  "learning_rate": 0.14490677803143556,
  "n_estimators": 950,
  "subsample": 0.9675996210280686,
  "colsample_bytree": 0.7466959369997275,
  "colsample_bylevel": 0.6008573652470146,
  "reg_alpha": 8.734427152096292e-05,
  "reg_lambda": 3.04526005837922e-05,
  "scale_pos_weight": 6.456521739130435,
  "random_state": 42,
  "n_jobs": -1,
  "eval_metric": "logloss"
}


## Step 5: Train Final Model on Full Training Set

In [13]:
# Cell for Training Final Model in Notebook 05
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
best_params["scale_pos_weight"] = n_neg / max(n_pos, 1)
best_params["random_state"] = config.RANDOM_STATE
best_params["n_jobs"] = -1
best_params["eval_metric"] = "logloss"

# FIX: Pass y_train explicitly to build_model
final_model = build_model("XGBoost", y_train=y_train, **best_params)

X_train_safe = xgb_safe_frame(X_train_final)
final_model.fit(X_train_safe, y_train)

# Sanity Check
train_pred = final_model.predict_proba(X_train_safe)[:, 1]
train_auc = roc_auc_score(y_train, train_pred)
logger.info(f"Training AUC (sanity check): {train_auc:.4f}")

# Save Artifacts IMMEDIATELY after training
joblib.dump(final_model, config.MODELS_DIR / "best_model_xgboost.joblib")
print("✅ Model trained and saved successfully.")

2026-09-01 09:56:16 | INFO     | prostate_bcr | Built model: XGBoost
2026-09-01 09:56:17 | INFO     | prostate_bcr | Training AUC (sanity check): 1.0000


✅ Model trained and saved successfully.


## Step 6: Save Artifacts

In [14]:
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Save the trained model
joblib.dump(final_model, config.MODELS_DIR / "best_model_xgboost.joblib")

# Save the fitted selector for transforming test data
joblib.dump(fitted_selector, config.MODELS_DIR / "fitted_selector.joblib")

# CRITICAL: Transform and save test data
X_test = pd.read_csv(config.PROCESSED_DIR / "X_test_preprocessed.csv", index_col=0)
y_test_df = pd.read_csv(config.PROCESSED_DIR / "y_test.csv")
y_test = y_test_df.iloc[:, 0] if len(y_test_df.columns) == 1 else y_test_df["BCR"]

# Apply the same transformation to test data
X_test_selected = transform_selected(X_test, fitted_selector)

# Ensure test data has same features as training
available_in_test = [f for f in final_features if f in X_test_selected.columns]
X_test_final = X_test_selected[available_in_test].copy()

# Fill missing features with 0.0
missing_feats = set(final_features) - set(available_in_test)
if missing_feats:
    for feat in missing_feats:
        X_test_final[feat] = 0.0
    X_test_final = X_test_final[final_features]

# Save transformed test data
X_test_final.to_csv(config.PROCESSED_DIR / "X_test_selected.csv")
y_test.to_csv(config.PROCESSED_DIR / "y_test.csv")

print("Model training complete. Artifacts saved:")
print(f"  - {config.MODELS_DIR / 'best_model_xgboost.joblib'}")
print(f"  - {config.MODELS_DIR / 'fitted_selector.joblib'}")
print(f"  - {config.PROCESSED_DIR / 'X_test_selected.csv'}")
print(f"  - {config.PROCESSED_DIR / 'y_test.csv'}")

Model training complete. Artifacts saved:
  - D:\Prostate_BCR\core\outputs\models\best_model_xgboost.joblib
  - D:\Prostate_BCR\core\outputs\models\fitted_layer1_selector.joblib


In [15]:
# ============================================================================
# Step 6: Prepare and Save Test Data for Final Evaluation
# ============================================================================
import joblib
from src.feature_selection import create_extended_engineered_features

config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

try:
    # 1. Load Raw Test Data
    X_test_raw = pd.read_csv(config.PROCESSED_DIR / "X_test_preprocessed.csv", index_col=0)
    y_test_df = pd.read_csv(config.PROCESSED_DIR / "y_test.csv")
    y_test = y_test_df.iloc[:, 0] if len(y_test_df.columns) == 1 else y_test_df['BCR']
    
    logger.info(f"Raw test data loaded: {X_test_raw.shape}")

    # 2. Apply Feature Engineering Safely
    # Use strict_mode=False to handle missing genes in test set gracefully
    X_test_eng, _ = create_extended_engineered_features(
        X_test_raw, 
        selected_genes=fitted_l1["mi_features"],
        correlation_threshold=0.90
    )

    # 3. Filter to Match Training Features & Handle Missing Columns
    # Get the exact features the model was trained on (from Notebook 04 output)
    final_features = pd.read_csv(config.TABLES_DIR / "selected_features_final.csv")["feature"].tolist()
    
    available_cols = [f for f in final_features if f in X_test_eng.columns]
    missing_cols = set(final_features) - set(available_cols)
    
    if missing_cols:
        logger.warning(f"Filling {len(missing_cols)} missing features in test set with 0.0: {list(missing_cols)[:5]}...")
        for col in missing_cols:
            X_test_eng[col] = 0.0
            
    # Reorder columns to exactly match training data
    X_test_final = X_test_eng[final_features].copy()
    
    # 4. Save Artifacts
    X_test_final.to_csv(config.PROCESSED_DIR / "X_test_selected.csv", index=False)
    y_test.to_csv(config.PROCESSED_DIR / "y_test.csv", index=False)
    
    logger.info(f"✅ Test data saved successfully: {X_test_final.shape}")
    print(f"   - X_test_selected.csv ({X_test_final.shape})")
    print(f"   - y_test.csv ({len(y_test)} samples)")

except Exception as e:
    # CATCH ALL ERRORS TO PREVENT SILENT FAILURE
    logger.error(f"❌ FAILED to prepare/save test data: {type(e).__name__}: {e}")
    raise  # Re-raise to stop execution and show traceback

2026-09-01 09:56:18 | INFO     | prostate_bcr | Raw test data loaded: (86, 19018)
2026-09-01 09:56:18 | INFO     | prostate_bcr | Layer 2 - Created 4 clean engineered features
2026-09-01 09:56:18 | INFO     | prostate_bcr | ✅ Test data saved successfully: (86, 50)


   - X_test_selected.csv ((86, 50))
   - y_test.csv (86 samples)
